# Miro Face Detector - CNN Model Training

This notebook trains a Convolutional Neural Network (CNN) to classify images as either 'Miro' (1) or 'Random' (0). The data should be preprocessed by `src/build_dataset.py` before running this notebook. It is designed to be executable in Google Colab as requested.

## Configuration

Ensure your `config.yaml` file is properly positioned before training. For example, if the drive mounted fine and the location of the dataset is at `/content/drive/MyDrive/processed`, place the config file there or at the project root.

Example `config.yaml` content:
```yaml
model:
  img_size: 128
  output_path: /content/drive/MyDrive/processed/models/miro_detector.keras
data:
  dataset_csv: /content/drive/MyDrive/processed/dataset.csv
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

yaml_path = '../config.yaml'
if os.path.exists(yaml_path):
    import yaml
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    DATA_CSV = '../' + config['data']['dataset_csv']
    IMG_SIZE = config['model']['img_size']
    MODEL_OUTPUT = '../' + config['model']['output_path']
    DATA_DIR = '../'
else:
    # Fallback for Colab execution where structure might differ
    DATA_CSV = '/content/drive/MyDrive/processed/dataset.csv'
    IMG_SIZE = 128
    MODEL_OUTPUT = '/content/drive/MyDrive/models/miro_detector.keras'
    DATA_DIR = '/content/drive/MyDrive/'

# Ensure we use the modern keras extension to avoid warnings
if MODEL_OUTPUT.endswith('.h5'):
    MODEL_OUTPUT = MODEL_OUTPUT[:-3] + '.keras'

print(f"TensorFlow Version: {tf.__version__}")

## 1. Data Loading and Preparation

In [ ]:
import concurrent.futures

df = pd.read_csv(DATA_CSV)

def process_image(filepath, label):
    filepath = str(filepath).replace(chr(92), '/')
    img_path = os.path.join(DATA_DIR, filepath)
    
    if not os.path.exists(img_path) and 'data/processed/' in filepath:
        alt_path = os.path.join('/content/drive/MyDrive/processed/', filepath.split('data/processed/')[-1])
        if os.path.exists(alt_path):
            img_path = alt_path

    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return img, label, None
    else:
        return None, None, img_path

def load_images_from_df(dataframe):
    images = []
    labels = []
    error_count = 0
    loaded_count = 0
    
    paths = dataframe['filepath'].tolist()
    df_labels = dataframe['label'].tolist()
    total_images = len(paths)
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        results = executor.map(process_image, paths, df_labels)
       
        for img, label, err_path in results:
            if img is not None:
                images.append(img)
                labels.append(label)
                loaded_count += 1
                if loaded_count % 10 == 0:
                    print(f"Loaded {loaded_count} out of {total_images} images...")
            else:
                error_count += 1
                if error_count <= 5:
                    print(f"Error loading image: {err_path}")
                elif error_count == 6:
                    print("... additional image load errors hidden.")
                
    return np.array(images) / 255.0, np.array(labels)

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

print("Loading training data...")
X_train, y_train = load_images_from_df(train_df)
print("Loading testing data...")
X_test, y_test = load_images_from_df(test_df)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

if len(X_train) == 0:
    raise ValueError("No images loaded. Halting execution to prevent Keras shape errors. Please verify DATA_DIR and filepath alignment.")

## 2. Model Architecture

Building a custom CNN following the assignment specification: Convolutional -> Max Pooling -> Flatten -> Dense.

In [ ]:
model = models.Sequential([
    # Input Layer to resolve Keras warnings
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Convolutional & Pooling Layer 1
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Convolutional & Pooling Layer 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Convolutional & Pooling Layer 3
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flatten 2D feature maps to 1D vector
    layers.Flatten(),

    # Dense Fully Connected layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid') # Binary classification
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

## 3. Training and Evaluation

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test),
    batch_size=32
)

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
plt.show()

## 4. Exporting Model Weights

In [ ]:
os.makedirs(os.path.dirname(MODEL_OUTPUT), exist_ok=True)
model.save(MODEL_OUTPUT)
print(f"Model saved to {MODEL_OUTPUT}")